# FinTrustX: Explainable AI for Credit Risk

**Notebook:** `05_Explainable_AI`  
This notebook explains the persisted champion without retraining, tuning, or modifying the held-out test set. All explainability logic is delegated to the reusable classes in `src/explainability/`. `TARGET = 1` denotes default.

> Explanations describe this model's associations in the processed data; they are not evidence that a feature causes default and are not, by themselves, lending decisions.

## 1. Environment setup

Run this notebook from the repository root (or the `notebooks/` directory). For normal portfolio execution, mount the full repository in Google Colab before running these cells.

In [ ]:
from __future__ import annotations

import importlib
import importlib.util
import os
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

def find_project_root(anchor_dirs=('src', 'data', 'models'), max_upward=4) -> Path:
    current = Path.cwd().resolve()
    if (current / 'Credit-risk-ai' / 'src').is_dir():
        return (current / 'Credit-risk-ai').resolve()
    for _ in range(max_upward):
        if all((current / marker).exists() for marker in anchor_dirs):
            return current
        if current.parent == current:
            break
        current = current.parent
    if (Path.cwd() / 'src').exists():
        return Path.cwd().resolve()
    if (Path.cwd().parent / 'src').exists():
        return Path.cwd().parent.resolve()
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import tensorflow as tf
    gpu_status = 'GPU available' if tf.config.list_physical_devices('GPU') else 'Running on CPU'
except ImportError:
    tf = None
    gpu_status = 'TensorFlow is unavailable; GPU status cannot be checked'

package_names = ['numpy', 'pandas', 'scikit-learn', 'shap', 'lime', 'dice-ml', 'tensorflow', 'matplotlib']
package_versions = {'Python': platform.python_version()}
for package_name in package_names:
    try:
        package_versions[package_name] = version(package_name)
    except PackageNotFoundError:
        package_versions[package_name] = 'MISSING'
display(pd.Series(package_versions, name='Version').to_frame())
print(f'Project root: {PROJECT_ROOT}')
print(gpu_status)
if importlib.util.find_spec('google.colab') is None:
    print('WARNING: Running locally. Use a Google Colab runtime for the expected notebook environment.')

from src.config import DEEP_LEARNING_TRAINING, MODEL_DIR, REPORT_DIR

required_explainability_modules = {
    'SHAP': ('src.explainability.shap_explainer', 'SHAPExplainerFactory'),
    'LIME': ('src.explainability.lime_explainer', 'LimeExplainer'),
    'DiCE': ('src.explainability.dice_explainer', 'DiceExplainer'),
    'Integrated Gradients': ('src.explainability.integrated_gradients', 'IntegratedGradientsExplainer'),
    'Report generator': ('src.explainability.report_generator', 'ReportGenerator'),
}
explainability_classes, module_status = {}, {}
for label, (module_name, class_name) in required_explainability_modules.items():
    try:
        explainability_classes[label] = getattr(importlib.import_module(module_name), class_name)
        module_status[label] = 'available'
    except Exception as error:
        module_status[label] = f'unavailable: {type(error).__name__}: {error}'
display(pd.Series(module_status, name='Import status').to_frame())
if module_status['SHAP'] != 'available' or module_status['LIME'] != 'available':
    raise ImportError('Required SHAP or LIME project module could not be imported; see the table above.')

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_colwidth', 120)
REPORT_OUTPUT_DIR = REPORT_DIR / 'explainability'
REPORT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load the final model

`src/config.py` provides `MODEL_DIR`; the classical-model notebook persisted the selected champion there as `best_model.joblib`. No model is fitted in this notebook.

In [ ]:
BEST_MODEL_PATH = MODEL_DIR / 'best_model.joblib'
if not BEST_MODEL_PATH.is_file():
    raise FileNotFoundError(f'Champion model not found: {BEST_MODEL_PATH}')

model = joblib.load(BEST_MODEL_PATH)
model_metadata = pd.DataFrame([{
    'model type': type(model).__name__,
    'model path': str(BEST_MODEL_PATH),
    'successfully loaded': model is not None,
}])
display(model_metadata)
assert hasattr(model, 'predict') and hasattr(model, 'predict_proba'), 'The champion must support predict and predict_proba.'


## 3. Load immutable held-out test data

The prior notebooks saved already transformed, numeric matrices. The training matrix is read only as background/reference data for the existing SHAP, LIME, and DiCE wrappers; it is never used to fit or tune a model here.

In [ ]:
TARGET_COLUMN = DEEP_LEARNING_TRAINING['target_column']
TRAIN_PATH = PROJECT_ROOT / 'data' / DEEP_LEARNING_TRAINING['train_filename']
TEST_PATH = PROJECT_ROOT / 'data' / DEEP_LEARNING_TRAINING['test_filename']
train_frame = pd.read_parquet(TRAIN_PATH)
test_frame = pd.read_parquet(TEST_PATH)
if TARGET_COLUMN not in train_frame or TARGET_COLUMN not in test_frame:
    raise KeyError(f'Missing target column {TARGET_COLUMN!r} in a processed dataset.')
X_train = train_frame.drop(columns=TARGET_COLUMN)
X_test = test_frame.drop(columns=TARGET_COLUMN)
y_test = test_frame[TARGET_COLUMN].astype(int).copy()
if list(X_train.columns) != list(X_test.columns):
    raise ValueError('Processed train/test feature schemas do not match.')
feature_names = X_test.columns.tolist()
model_feature_count = getattr(model, 'n_features_in_', None)
if model_feature_count is not None and X_test.shape[1] != model_feature_count:
    raise ValueError(f'Model expects {model_feature_count} features; X_test has {X_test.shape[1]}.')
display(pd.DataFrame([{
    'X_test rows': len(X_test), 'X_test features': X_test.shape[1],
    'model input features': model_feature_count if model_feature_count is not None else 'not exposed',
    'schema verified': model_feature_count is None or X_test.shape[1] == model_feature_count,
}]))


## 4. Representative applicants

The following are anonymous row positions only. No raw applicant attributes or identifiers are displayed. Categories missing from this held-out prediction set are skipped.

In [ ]:
test_probabilities = model.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= 0.5).astype(int)
outcomes = pd.DataFrame({'actual class': y_test.to_numpy(), 'predicted class': test_predictions, 'predicted probability': test_probabilities}, index=X_test.index)
selection_rules = {
    'Correctly predicted low-risk applicant': (outcomes['actual class'].eq(0) & outcomes['predicted class'].eq(0)),
    'Correctly predicted high-risk applicant': (outcomes['actual class'].eq(1) & outcomes['predicted class'].eq(1)),
    'False positive': (outcomes['actual class'].eq(0) & outcomes['predicted class'].eq(1)),
    'False negative': (outcomes['actual class'].eq(1) & outcomes['predicted class'].eq(0)),
}
selected_indices = {}
for label, mask in selection_rules.items():
    candidates = outcomes.loc[mask]
    if not candidates.empty:
        selected_indices[label] = candidates.index[0]
    else:
        print(f'Skipping {label}: no such held-out example exists.')
selected_applicants = outcomes.loc[list(selected_indices.values())].copy()
selected_applicants.insert(0, 'example type', list(selected_indices.keys()))
selected_applicants = selected_applicants.reset_index(drop=True)
display(selected_applicants.style.format({'predicted probability': '{:.3f}'}))
if not selected_indices:
    raise RuntimeError('No representative test examples were available.')
LOCAL_INDEX = next(iter(selected_indices.values()))


## 5. SHAP explanations

The project `SHAPExplainerFactory` selects the implementation. For the persisted Random Forest it selects `TreeSHAPExplainer`. A bounded background/reference sample keeps the global calculation practical. Identifier-like feature names are masked in all displayed explainability outputs.

In [ ]:
SHAPExplainerFactory = explainability_classes['SHAP']
safe_feature_names = ['[identifier excluded]' if 'SK_ID' in name.upper() else name for name in feature_names]
BACKGROUND_SIZE = min(200, len(X_train))
GLOBAL_SAMPLE_SIZE = min(500, len(X_test))
background_data = X_train.sample(BACKGROUND_SIZE, random_state=42).to_numpy()
global_sample = X_test.sample(GLOBAL_SAMPLE_SIZE, random_state=42)
shap_explainer = SHAPExplainerFactory.create(model, safe_feature_names, background_data=background_data)
shap_values = shap_explainer.compute_shap_values(global_sample.to_numpy())
print(f'Project explainer selected: {type(shap_explainer).__name__}')
print(f'SHAP output type: {type(shap_values).__name__}')

# The reusable module owns the SHAP calculation. This adapter only selects the positive/default class
# from the module output so its values can be displayed consistently across SHAP versions.
def positive_class_shap(values):
    raw = values.values if hasattr(values, 'values') else values
    if isinstance(raw, list):
        raw = raw[1] if len(raw) > 1 else raw[0]
    raw = np.asarray(raw)
    if raw.ndim == 3:
        raw = raw[:, :, 1] if raw.shape[-1] > 1 else raw[:, :, 0]
    if raw.ndim != 2:
        raise ValueError(f'Unsupported SHAP output shape: {raw.shape}')
    return raw

shap_matrix = positive_class_shap(shap_values)
global_importance = pd.DataFrame({
    'feature': safe_feature_names,
    'mean absolute SHAP value': np.abs(shap_matrix).mean(axis=0),
}).sort_values('mean absolute SHAP value', ascending=False)
display(global_importance.head(15).reset_index(drop=True))
plt.figure(figsize=(10, 7))
sns.barplot(data=global_importance.head(15), x='mean absolute SHAP value', y='feature', color='#4C72B0')
plt.title('Global SHAP importance — held-out sample')
plt.tight_layout()
global_shap_plot_path = REPORT_OUTPUT_DIR / 'shap_global_importance.png'
plt.savefig(global_shap_plot_path, dpi=300, bbox_inches='tight')
plt.show()

try:
    # Existing project visualization method; compatibility depends on the installed SHAP version.
    shap_explainer.summary_plot(global_sample.to_numpy())
    summary_plot_path = shap_explainer.output_dir / 'shap_summary_plot.png'
    print(f'Saved project SHAP summary plot: {summary_plot_path}')
except Exception as error:
    summary_plot_path = None
    print(f'Project SHAP summary plot is not compatible with this model/SHAP version: {type(error).__name__}: {error}')


In [ ]:
local_position = X_test.index.get_loc(LOCAL_INDEX)
local_values = X_test.loc[LOCAL_INDEX].to_numpy()
local_shap = shap_matrix[local_position]
local_shap_table = pd.DataFrame({
    'feature': safe_feature_names,
    'feature value': local_values,
    'SHAP value': local_shap,
})
local_shap_table['contribution direction'] = np.where(local_shap_table['SHAP value'] >= 0, 'increases predicted default risk', 'decreases predicted default risk')
local_shap_table = local_shap_table.reindex(local_shap_table['SHAP value'].abs().sort_values(ascending=False).index)
display(local_shap_table.head(15).reset_index(drop=True))
plt.figure(figsize=(10, 7))
local_plot = local_shap_table.head(15).sort_values('SHAP value')
plt.barh(local_plot['feature'], local_plot['SHAP value'], color=np.where(local_plot['SHAP value'] >= 0, '#C44E52', '#55A868'))
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Local SHAP contributions — anonymous applicant')
plt.xlabel('Contribution toward default probability')
plt.tight_layout()
local_shap_plot_path = REPORT_OUTPUT_DIR / 'shap_local_importance.png'
plt.savefig(local_shap_plot_path, dpi=300, bbox_inches='tight')
plt.show()


## 6. LIME explanation

`LimeExplainer` is used directly from `src/explainability/`. It supports models exposing `predict_proba`, which the champion does.

In [ ]:
LimeExplainer = explainability_classes['LIME']
lime_explainer = LimeExplainer(
    model=model, X_train=X_train.to_numpy(), feature_names=safe_feature_names,
    output_dir=REPORT_OUTPUT_DIR / 'lime',
)
lime_explanation = lime_explainer.explain_instance(local_values, num_samples=5000)
lime_results = pd.DataFrame(lime_explainer.feature_importance(lime_explanation), columns=['feature condition', 'contribution weight'])
lime_results['contribution direction'] = np.where(lime_results['contribution weight'] >= 0, 'increases selected-class score', 'decreases selected-class score')
display(lime_results)
lime_html_path = lime_explainer.save_html(lime_explanation, 'local_lime_explanation.html')
lime_png_path = lime_explainer.save_png(lime_explanation, 'local_lime_explanation.png')
print(f'Saved LIME HTML: {lime_html_path}')
print(f'Saved LIME plot: {lime_png_path}')


## 7. DiCE counterfactual explanation

The existing `DiceExplainer` is attempted against the persisted scikit-learn model and processed reference data. It may be unavailable if `dice-ml` is not installed or cannot support this model/data representation. Counterfactuals are synthetic suggestions; the real test set is never changed, and no financial realism is implied unless constraints are explicitly supplied.

In [ ]:
dice_results = None
dice_artifacts = []
if module_status['DiCE'] != 'available':
    print(f'DiCE is unavailable: {module_status["DiCE"]}')
else:
    try:
        DiceExplainer = explainability_classes['DiCE']
        dice_reference_data = X_train.copy()
        dice_reference_data[TARGET_COLUMN] = train_frame[TARGET_COLUMN].astype(int).to_numpy()
        dice_explainer = DiceExplainer(
            model=model, dataframe=dice_reference_data, continuous_features=feature_names,
            target_name=TARGET_COLUMN, output_dir=REPORT_OUTPUT_DIR / 'dice',
        )
        dice_query = X_test.loc[[LOCAL_INDEX]].copy()
        counterfactuals = dice_explainer.generate_counterfactuals(dice_query, total_cfs=3, desired_class='opposite')
        counterfactual_frame = counterfactuals.cf_examples_list[0].final_cfs_df.copy()
        cf_features = counterfactual_frame[feature_names]
        cf_probabilities = model.predict_proba(cf_features)[:, 1]
        changes = []
        for _, counterfactual in cf_features.iterrows():
            changed = counterfactual.index[counterfactual.ne(X_test.loc[LOCAL_INDEX])]
            for feature in changed:
                if 'SK_ID' not in feature.upper():
                    changes.append({'feature': feature, 'original value': X_test.loc[LOCAL_INDEX, feature], 'counterfactual value': counterfactual[feature]})
        dice_results = pd.DataFrame(changes)
        counterfactual_overview = pd.DataFrame({
            'original prediction': [int(test_predictions[local_position])] * len(cf_features),
            'original probability': [float(test_probabilities[local_position])] * len(cf_features),
            'counterfactual prediction': (cf_probabilities >= 0.5).astype(int),
            'counterfactual probability': cf_probabilities,
        })
        display(counterfactual_overview.style.format({'original probability': '{:.3f}', 'counterfactual probability': '{:.3f}'}))
        display(dice_results)
        dice_artifacts = [dice_explainer.save_csv(counterfactuals), dice_explainer.save_json(counterfactuals)]
        print('DiCE constraints were not supplied by the current notebook; these changes are not guaranteed financially realistic.')
    except Exception as error:
        print(f'DiCE could not generate counterfactuals for the loaded model: {type(error).__name__}: {error}')


## 8. Neural-network explainability

`IntegratedGradientsExplainer` exists in `src/explainability/`. It is applicable only if the configured neural-network artifact is present and has the same processed input schema; it is not substituted for the selected Random Forest explanation.

In [ ]:
ig_artifacts = []
neural_model_path = MODEL_DIR / DEEP_LEARNING_TRAINING['model_filename']
if module_status['Integrated Gradients'] != 'available':
    print('Neural-network-specific explainability is not currently available because the project Integrated Gradients module could not be imported.')
elif tf is None or not neural_model_path.is_file():
    print('Neural-network-specific explainability is not currently implemented for this run: the configured neural-network model artifact is unavailable.')
else:
    neural_model = tf.keras.models.load_model(neural_model_path)
    if neural_model.input_shape[-1] != X_test.shape[1]:
        print('Integrated Gradients skipped: neural-network input feature count does not match processed test data.')
    else:
        IntegratedGradientsExplainer = explainability_classes['Integrated Gradients']
        ig_explainer = IntegratedGradientsExplainer(neural_model, output_dir=REPORT_OUTPUT_DIR / 'integrated_gradients')
        ig_input = pd.DataFrame([local_values], columns=safe_feature_names)
        ig_results = ig_explainer.explain(ig_input)
        display(ig_results.head(15))
        ig_explainer.visualize(ig_results, top_n=15)
        ig_artifacts = [ig_explainer.save_csv(ig_results), ig_explainer.save_json(ig_results)]


## 9. Compare available explainability methods

In [ ]:
comparison = pd.DataFrame([
    ['SHAP', 'Tree, Keras, and fallback models via project factory', 'Feature attributions', 'Global and local', 'SHAP values and plots', 'Model review and feature-level investigation'],
    ['LIME', 'Models with predict_proba or Keras predict', 'Local surrogate explanation', 'Local', 'Feature-condition weights', 'Individual prediction review'],
    ['DiCE', 'Scikit-learn or TensorFlow models supported by dice-ml', 'Counterfactual examples', 'Local', 'Synthetic alternative records', 'Explore possible model-score changes'],
    ['Integrated Gradients', 'TensorFlow/Keras neural networks', 'Gradient-based attributions', 'Local', 'Feature importance table', 'Neural-network prediction review'],
], columns=['Method', 'Model supported', 'Explanation type', 'Global/Local', 'Output', 'Use case'])
available_methods = {'SHAP', 'LIME'} | ({'DiCE'} if module_status['DiCE'] == 'available' else set()) | ({'Integrated Gradients'} if module_status['Integrated Gradients'] == 'available' else set())
display(comparison[comparison['Method'].isin(available_methods)].reset_index(drop=True))


## 10. Business interpretation

The following evidence is model-specific. Positive SHAP values move the model toward the default class, while negative values move it away. Contributions should be reviewed alongside data quality, policy constraints, calibration, fairness testing, and human judgment.

In [ ]:
risk_increasing = local_shap_table[local_shap_table['SHAP value'] > 0].head(5)
risk_decreasing = local_shap_table[local_shap_table['SHAP value'] < 0].head(5)
display(Markdown('**Largest model contributions increasing predicted default risk**'))
display(risk_increasing[['feature', 'SHAP value']])
display(Markdown('**Largest model contributions decreasing predicted default risk**'))
display(risk_decreasing[['feature', 'SHAP value']])
if dice_results is not None and not dice_results.empty:
    display(Markdown('DiCE produced synthetic feature changes that may alter the model score. They are not causal advice or guaranteed financially feasible changes.'))
else:
    display(Markdown('No DiCE result was available for this environment/model combination; no counterfactual business conclusion is made.'))


## 11. Save explainability results

The existing `ReportGenerator` persists tabular artifacts under `reports/explainability/`; plot and HTML paths are collected below.

In [ ]:
ReportGenerator = explainability_classes['Report generator']
report_generator = ReportGenerator(output_dir=REPORT_OUTPUT_DIR)
saved_artifacts = [
    report_generator.save_csv(global_importance.reset_index(drop=True), 'shap_global_importance.csv'),
    report_generator.save_csv(local_shap_table.reset_index(drop=True), 'shap_local_contributions.csv'),
    report_generator.save_csv(lime_results, 'lime_local_contributions.csv'),
    report_generator.save_json(local_shap_table.reset_index(drop=True), 'shap_local_contributions.json'),
    global_shap_plot_path, local_shap_plot_path, lime_html_path, lime_png_path,
] + dice_artifacts + ig_artifacts
if summary_plot_path is not None:
    saved_artifacts.append(summary_plot_path)
artifact_table = pd.DataFrame({'artifact path': [str(path) for path in saved_artifacts], 'exists': [Path(path).exists() for path in saved_artifacts]})
display(artifact_table)
assert artifact_table['exists'].all(), 'One or more explainability artifacts were not saved.'


## 12. Final summary

- **Best model explained:** persisted `RandomForestClassifier` champion from `models/best_model.joblib`.
- **Most important features:** inspect the saved global SHAP ranking; it reflects the sampled held-out data, not causal importance.
- **SHAP:** provides global and local signed feature contributions for the model’s default-risk score.
- **LIME:** provides a local, surrogate-based feature-condition explanation for the anonymous selected record.
- **DiCE:** when available, presents unconstrained synthetic alternatives; they are not guaranteed feasible or fair recommendations.
- **Neural networks:** Integrated Gradients is available in the source tree, but is run only when the configured Keras artifact exists and its schema matches.
- **Limitations:** processed one-hot/scaled features can be harder to interpret than source variables; SHAP/LIME can disagree; counterfactuals depend on data and constraints; selection and explanations require stability, calibration, subgroup-fairness, proxy-feature, privacy, and adverse-action review before any lending use.